# Clase 9 — Regresión y taller de diagnóstico en ML

En esta clase damos un nuevo paso dentro del recorrido de ML del módulo. Primero, aprendemos a resolver problemas de **regresión** (predecir un número en lugar de una categoría). Después, cerramos con un **taller grupal de diagnóstico** donde los equipos analizan situaciones problemáticas, detectan errores conceptuales y se apoyan en la inteligencia artificial para corregirlos y justificar sus decisiones.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Regresión vs. clasificación |
| 2 | El dataset: precios de casas |
| 3 | Modelo lineal y métricas de regresión |
| 4 | Ridge y Lasso: regularización |
| 5 | Detectar sobreajuste |
| 6 | Taller grupal: diagnosticar decisiones de ML |
| 7 | Casos para analizar y corregir |
| 8 | Preparación de la presentación para la clase 10 |

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

print("Librerías listas.")

Librerías listas.


---
## 1. Regresión vs. clasificación

| | Clasificación | Regresión |
|---|---|---|
| **Salida del modelo** | Categoría discreta | Número continuo |
| **Pregunta típica** | ¿Es spam o no? | ¿Cuánto vale? |
| **Métrica principal** | Accuracy / F1 | MSE / RMSE / R² |
| **Ejemplo** | ¿El cliente va a abandonar? | ¿Cuánto va a consumir? |

_
> 💡 El flujo de trabajo es idéntico: cargar datos → split → preprocesar → entrenar → evaluar. Solo cambian el tipo de modelo y las métricas de evaluación.

---
## 2. El dataset: precios de casas en California

El dataset California Housing contiene datos de 20,640 zonas del estado de California. La tarea es predecir el valor mediano de las casas (`MedHouseVal`, en cientos de miles de dólares).

In [ ]:
# ─── Cargar el dataset ────────────────────────────────────────────────────────
from urllib.error import URLError

try:
    housing = fetch_california_housing(as_frame=True)
except URLError as exc:
    if "CERTIFICATE_VERIFY_FAILED" not in str(exc):
        raise

    import ssl
    import certifi

    # Reintento usando el bundle de certificados de certifi.
    ssl._create_default_https_context = (
        lambda: ssl.create_default_context(cafile=certifi.where())
    )
    housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Shape: {df.shape}")
print()
print("Descripción de las variables:")

descripciones = {
    "MedInc":      "Ingreso mediano del hogar (en 10k USD)",
    "HouseAge":    "Edad mediana de las casas",
    "AveRooms":    "Promedio de habitaciones por hogar",
    "AveBedrms":   "Promedio de dormitorios por hogar",
    "Population":  "Población del bloque",
    "AveOccup":    "Promedio de ocupantes por hogar",
    "Latitude":    "Latitud del bloque",
    "Longitude":   "Longitud del bloque",
    "MedHouseVal": "Valor mediano de la casa — ETIQUETA (objetivo)",
}
for col, desc in descripciones.items():
    print(f"  {col:<15}: {desc}")

In [ ]:
# ─── Exploración básica ───────────────────────────────────────────────────────
print("Estadísticas descriptivas:")
df.describe().round(2)

In [ ]:
# ─── Distribución del target ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(df["MedHouseVal"], bins=50, color="steelblue", edgecolor="white")
ax1.set(xlabel="Valor mediano (100k USD)", ylabel="Frecuencia",
        title="Distribución del target")

ax2.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.1, s=3, color="coral")
ax2.set(xlabel="Ingreso mediano", ylabel="Valor casa",
        title="Ingreso vs. Precio (feature más correlacionada)")

plt.tight_layout()
plt.show()

In [ ]:
# ─── Split ────────────────────────────────────────────────────────────────────
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} — Test: {X_test.shape[0]}")

---
## 3. Modelo lineal y métricas de regresión

### ¿Cómo medimos el éxito en una Regresión?

A diferencia de la clasificación (donde medimos si "acertamos" o "fallamos" con Accuracy o F1), en regresión trabajamos con valores continuos. Rara vez el modelo le pegará al valor exacto (ej. predecir exactamente $250,501.23). Por eso, lo que medimos es **qué tan lejos** nos quedamos de la realidad.

#### 1. Error Cuadrático Medio (MSE) y RMSE
El **MSE (Mean Squared Error)** mide el promedio de los errores al cuadrado.
- **¿Por qué al cuadrado?**
    1. Para que los errores negativos y positivos no se cancelen entre sí.
    2. Para **penalizar más fuerte los errores grandes**. Un error de 10 unidades "duele" como 100 ($10^2$), mientras que uno de 2 "duele" solo 4 ($2^2$).
- **RMSE (Root MSE):** Es simplemente la raíz del MSE. Se usa para volver a las unidades originales (en nuestro caso, cientos de miles de dólares).

#### 2. Coeficiente de Determinación ($R^2$)
El $R^2$ (R-cuadrado) nos dice qué tan bueno es nuestro modelo comparado con un "modelo bobo" que solo predijera siempre el promedio de todos los precios.
- **R² = 1:** Modelo perfecto (predice todo sin error).
- **R² = 0:** El modelo no es mejor que decir siempre el promedio.
- **Interpretación:** Si el $R^2$ es 0.70, significa que nuestro modelo explica el 70% de la variabilidad de los precios de las casas usando los datos que le dimos.

In [ ]:
# ─── Pipeline: scaler + regresión lineal ─────────────────────────────────────
pipeline_lr = Pipeline([
    ("escalar", StandardScaler()),
    ("modelo",  LinearRegression()),
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)

print("Métricas — Regresión Lineal:")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_lr):.4f}  (error absoluto promedio en 100k USD)")
print(f"  RMSE: {root_mean_squared_error(y_test, y_pred_lr):.4f}  (igual que MAE pero penaliza errores grandes)")
print(f"  R²:   {r2_score(y_test, y_pred_lr):.4f}  (1.0 = perfecto, 0 = tan malo como la media)")

| Métrica | Fórmula | Qué mide |
|---|---|---|
| **MAE** | $\frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | Error promedio en las mismas unidades que el target |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Error promedio con mayor peso a los errores grandes |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporción de la varianza explicada por el modelo |

In [ ]:
# ─── Visualizar predicciones vs. valores reales ───────────────────────────────
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred_lr, alpha=0.3, s=5, color="steelblue")

# Línea perfecta: si el modelo fuera perfecto, todos los puntos caerían acá
lim = [y_test.min(), y_test.max()]
plt.plot(lim, lim, color="red", linewidth=1, label="Predicción perfecta")

plt.xlabel("Valor real")
plt.ylabel("Valor predicho")
plt.title("Regresión Lineal: real vs. predicho")
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Regularización: ¿Cómo evitar que el modelo "delire"?

A veces, la regresión lineal se vuelve demasiado "ambiciosa" y asigna importancias exageradas a ciertas variables, especialmente cuando hay muchas o están muy relacionadas entre sí. Esto lleva al **sobreajuste**.

Para evitarlo, usamos técnicas de **Regularización**, que básicamente le ponen una "multa" o penalización al modelo si intenta usar coeficientes demasiado grandes.

### Ridge (L2)
- **La multa:** Se basa en el cuadrado de los coeficientes.
- **Efecto:** Obliga a todos los coeficientes a ser pequeños, pero no los elimina. Es excelente para cuando todas las variables aportan un poco.

### Lasso (L1)
- **La multa:** Se basa en el valor absoluto de los coeficientes.
- **Efecto:** Es más agresiva; puede llevar los coeficientes de las variables menos importantes **exactamente a cero**. 
- **Bonus:** Funciona como un selector automático de variables (quita las que no sirven).

| Modelo | Penalización | Filosofía |
|---|---|---|
| **Ridge** | $\lambda \sum w_i^2$ | "Que todos ayuden, pero poquito" |
| **Lasso** | $\lambda \sum \lvert w_i \rvert$ | "Solo los que valen la pena se quedan" |

In [ ]:
# ─── Comparar Ridge, Lasso y Linear ──────────────────────────────────────────
modelos_reg = {
    "Linear":        LinearRegression(),
    "Ridge (α=1)": Ridge(alpha=1.0),
    "Lasso (α=0.1)": Lasso(alpha=0.1, max_iter=5000),
}

resultados_reg = []
for nombre, modelo in modelos_reg.items():
    pipe = Pipeline([("escalar", StandardScaler()), ("modelo", modelo)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    resultados_reg.append({
        "modelo":  nombre,
        "MAE":     mean_absolute_error(y_test, y_pred),
        "RMSE":    root_mean_squared_error(y_test, y_pred),
        "R²":      r2_score(y_test, y_pred),
    })

df_reg = pd.DataFrame(resultados_reg).set_index("modelo")
df_reg.round(4)

---
## 5. Detectar sobreajuste

El **sobreajuste (overfitting)** ocurre cuando el modelo aprende los datos de train demasiado bien y no generaliza. Se detecta comparando el score en train vs. test: si el train es mucho mejor, hay overfitting.

In [ ]:
# ─── Train vs. Test score para detectar overfitting ──────────────────────────
print("R² en train vs. test:")
print()

for nombre, modelo in modelos_reg.items():
    pipe = Pipeline([("escalar", StandardScaler()), ("modelo", modelo)])
    pipe.fit(X_train, y_train)

    r2_train = r2_score(y_train, pipe.predict(X_train))
    r2_test  = r2_score(y_test,  pipe.predict(X_test))
    diferencia = r2_train - r2_test

    flag = "⚠️  posible overfitting" if diferencia > 0.05 else "✓"
    print(f"  {nombre:<18}  train={r2_train:.4f}  test={r2_test:.4f}  diff={diferencia:.4f}  {flag}")

> 💡 En modelos lineales con datasets grandes, el overfitting es raro. Aparece más frecuentemente en Decision Trees sin profundidad limitada o en modelos con muchas features y pocos datos.

---
## 6. Taller grupal: diagnosticar decisiones de ML

Hasta ahora construimos una secuencia muy concreta:
- en la clase 6 entendimos **qué problemas resuelve ML** y qué familias de algoritmos existen;
- en la clase 7 vimos el **flujo estándar** de trabajo;
- en la clase 8 comparamos modelos de **clasificación** con evidencia;
- en esta clase vimos **regresión** y cómo cambia la evaluación cuando el objetivo es numérico.

El ejercicio final de la clase no consiste en programar desde cero ni en construir un proyecto técnico completo. El foco está en **afinar el criterio** para reconocer qué decisiones tienen sentido y cuáles no.

Van a trabajar en grupos sobre situaciones ya resueltas, algunas con incoherencias deliberadas. La tarea del grupo es detectar qué está mal, proponer una corrección y justificarla con conceptos del módulo.

 > 💡 La inteligencia artificial se usa acá como herramienta de apoyo: puede ayudar a aclarar conceptos, comparar alternativas y mejorar la explicación del grupo, pero no reemplaza la decisión final.

### Cómo trabajar en grupo con apoyo de IA

 En cada caso, el grupo debe seguir este procedimiento:
 1. Leer la situación e identificar cuál es el problema que se quiere resolver.
 2. Detectar la incoherencia: puede estar en el tipo de problema, el algoritmo elegido,
    la métrica, la interpretación de resultados o la lectura de sobreajuste.
 3. Discutir una corrección usando lo visto en las clases 6, 7, 8 y 9.
 4. Consultar una herramienta de IA para pedir explicaciones, contrastar opciones    o reformular la justificación del grupo.
 5. Escribir una respuesta final donde quede claro qué estaba mal, cómo lo corregirían y por qué.

#### Usen la IA para preguntas como estas:
 - "Explicame en lenguaje simple la diferencia entre clasificación y regresión"
 - "¿Cuándo conviene priorizar recall sobre accuracy?"
 - "¿Por qué un árbol muy profundo puede sobreajustar?"
 - "Ayudame a mejorar esta justificación sin cambiar su sentido"

> No usen la IA como reemplazo de la discusión. La respuesta del grupo debe ser una decisión argumentada por ustedes.

---
## 6. Kit de Supervivencia para el Diagnóstico

Antes de pasar al taller, repasemos los conceptos clave que hemos visto en estas últimas clases. Úsenlo como referencia para sus análisis:

### A. ¿Qué tarea estamos resolviendo?
- **Clasificación:** La respuesta es una "etiqueta" o categoría (Sí/No, Spam/No Spam, Perro/Gato).
- **Regresión:** La respuesta es un número continuo (Precio, Temperatura, Edad, Tiempo).

### B. ¿Cómo sabemos si va bien? (Métricas)
- **En Clasificación:**
    - *Accuracy:* ¿Cuánto acerté del total? (Cuidado si las clases están desbalanceadas).
    - *Recall:* ¿Cuántos de los casos "importantes" (ej. enfermos) logré encontrar?
    - *F1-Score:* El equilibrio entre precisión y recall.
- **En Regresión:**
    - *MAE/RMSE:* ¿Qué tan lejos (en promedio) estoy del valor real?
    - *$R^2$:* ¿Qué tanto mejor soy que simplemente decir el promedio?

### C. El flujo de trabajo
1. **Preprocesamiento:** ¿Hay variables con escalas muy distintas? (Ej. Edad 0-100 vs Salario 0-1.000.000). Si usamos algoritmos basados en distancia (como KNN), **hay que escalar**.
2. **Train/Test Split:** Nunca evaluamos el modelo con los mismos datos que usamos para enseñarle.
3. **Diagnóstico de Overfitting:**
    - *Train alto, Test bajo:* El modelo memorizó (Sobreajuste).
    - *Train bajo, Test bajo:* El modelo no aprendió nada (Subajuste).
    - *Train y Test parecidos y aceptables:* ¡El modelo generaliza bien!

---

## 7. Casos para analizar y corregir

### Caso 1 - Hospital y métrica mal elegida
Un hospital quiere construir un sistema para identificar pacientes con alto riesgo de complicaciones graves en las primeras 24 horas.

Un equipo propone lo siguiente:
- problema: clasificación
- modelo elegido: Logistic Regression
- métrica principal: accuracy
- argumento: "si la accuracy es alta, el sistema ya sirve"

Preguntas guía:
1. ¿La tarea está bien identificada como clasificación o no?
2. ¿La accuracy es suficiente en este contexto?
3. ¿Qué tipo de error sería más costoso para el hospital?
4. ¿Qué métrica mirarían con más atención y por qué?

### Caso 2 - Inmobiliaria y algoritmo incorrecto
Una inmobiliaria quiere estimar el precio de publicación de una vivienda usando superficie, barrio, antigüedad y cantidad de ambientes.

Un equipo propone lo siguiente:
- problema: clasificación
- modelo elegido: DecisionTreeClassifier
- métrica principal: accuracy
- argumento: "como los árboles son fáciles de interpretar, este modelo es ideal"

Preguntas guía:
1. ¿Qué tipo de problema es realmente?
2. ¿Tiene sentido usar un clasificador para este objetivo?
3. ¿Qué familia de modelos sería más coherente con la tarea?
4. ¿Qué métricas de regresión usarían para evaluarlo?

### Caso 3 - KNN sin pensar en la escala
Un equipo quiere clasificar clientes según si abandonarán o no un servicio. Para eso elige KNN y usa variables como edad, ingresos, cantidad de reclamos y meses de antigüedad.

El equipo aclara que decidió no escalar porque "los datos ya están en una tabla y el algoritmo puede aprender igual".

Preguntas guía:
1. ¿Qué problema hay en usar KNN sin revisar la escala de las variables?
2. ¿Qué variables podrían dominar la distancia sin que eso tenga sentido?
3. ¿Qué parte del flujo estándar de la clase 7 está faltando acá?
4. ¿Cómo corregirían conceptualmente esta propuesta?

### Caso 4 - Sobreajuste ignorado
Un equipo entrena un Decision Tree para clasificar reclamos ciudadanos. Obtiene estos resultados:
- score en train: 1.00
- score en test: 0.71

Igual concluye: "es el mejor modelo porque en entrenamiento aprendió perfecto".

Preguntas guía:
1. ¿Qué señal aparece en estos resultados?
2. ¿Por qué mirar solo train puede llevar a una mala decisión?
3. ¿Qué significa sobreajuste en este caso?
4. ¿Qué otro modelo o ajuste considerarían para mejorar la generalización?

---
## 8. Preparación para la clase 10

Cada grupo debe elegir uno de los casos y preparar una breve presentación para la próxima clase.

La presentación debe responder, como mínimo, estas cuatro preguntas:
1. ¿Cuál era el problema que se quería resolver?
2. ¿Qué incoherencia o error detectaron?
3. ¿Cómo lo corregirían usando los conceptos del módulo?
4. ¿En qué les ayudó la inteligencia artificial durante el análisis?